In [ ]:
#
!pip install datasets transformers
#

import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import pandas as pd

# 1. STREAMING THE HUGGING FACE FIN-DATASET
# Splitting off just a fraction ('train[:50000]') so it loads instantly into memory
print("Fetching 50,000 real financial rows...")
hf_dataset = load_dataset("mitulshah/global-financial-transaction-classifier", split='train[:50000]')

# Convert to Pandas instantly for clean verification
df = pd.DataFrame(hf_dataset)
print("\n--- Raw Ledger Head Sample ---")
print(df[['text', 'label']].head())

# 2. VECTORIZING PRODUCTION TEXT ENTRIES FOR PYTORCH
# Real ledgers rely on string data (e.g., 'Uber Trip', 'AWS Hosting').
# We build a simple vocabulary matrix mapping words to unique numeric tokens.
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=1000, stop_words='english')
X_vectorized = vectorizer.fit_transform(df['text']).toarray()
y_labels = df['label'].values

# 3. CREATING THE LARGE SCALE DATA LOADER
class ProductionLedgerDataset(Dataset):
    def __init__(self, features, targets):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create dataset container
large_dataset = ProductionLedgerDataset(X_vectorized, y_labels)

# Using multi-process workers to stream data into memory efficiently
large_dataloader = DataLoader(large_dataset, batch_size=64, shuffle=True)

print(f"\nReady for deep learning! Batches created: {len(large_dataloader)} (64 entries per batch)")